# FOSS Contribution Matchmaker
### Live Demo — Beyond the Unpredictable Loop

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR_USERNAME/foss_7_2026/blob/main/notebooks/foss_demo.ipynb)

> **Audience:** Run **Section 0** first (2 minutes). Then follow along.
> At **Section 3**, enter *your own* GitHub profile URL — you'll get a personalised plan live!

---

**The question this demo answers:**
> *"I want to contribute to open source — but where do I actually start?"*

**What we build:**
```
Your GitHub profile URL + interests
        │
        ▼
  LangGraph (orchestrator)
    ├─ analyse your GitHub profile
    ├─ [A2A] RepoFinderAgent     ─► searches GitHub for matching repos
    ├─ [A2A] ComplexityRaterAgent ─► rates each repo for your level
    └─ [A2A] OnboardingGuideAgent ─► smolagents CodeAgent inside!
                                      writes Python to fetch CONTRIBUTING.md + issues
        │
        ▼
  "Your First PR" personalised plan
```

---
# Section 0 — Setup
> **Run this first. Takes ~1-2 minutes.**

In [ ]:
!pip install -q "smolagents[toolkit]" litellm langgraph requests

In [ ]:
import os

def load_key(name: str):
    # 1. Already in environment (export GROQ_API_KEY=... before launching Jupyter)
    if os.environ.get(name):
        print(f'{name} loaded from environment'); return
    # 2. Colab Secrets — add via the key icon in the Colab left sidebar
    try:
        from google.colab import userdata
        os.environ[name] = userdata.get(name)
        print(f'{name} loaded from Colab Secrets'); return
    except Exception:
        pass
    # 3. .env file (local Jupyter) — create with: echo 'GROQ_API_KEY=...' > .env
    try:
        from dotenv import load_dotenv
        load_dotenv()
        if os.environ.get(name):
            print(f'{name} loaded from .env file'); return
    except ImportError:
        pass
    # 4. Prompt — input is masked and NOT saved in the notebook JSON
    import getpass
    os.environ[name] = getpass.getpass(f'{name}: ')

load_key('GROQ_API_KEY')

# Optional GitHub token for higher API rate limits during live demos
load_key('GITHUB_TOKEN') if 'GITHUB_TOKEN' not in os.environ else None

In [ ]:
# All imports + shared helper
import requests, json, uuid, base64, re
from dataclasses import dataclass, field
from typing import Any, Optional, List, TypedDict
from enum import Enum
from smolagents import CodeAgent, LiteLLMModel, tool
from langgraph.graph import StateGraph, END

model = LiteLLMModel(model_id="groq/llama-3.3-70b-versatile")

def gh_get(url, params=None):
    headers = {}
    if os.environ.get("GITHUB_TOKEN"):
        headers["Authorization"] = f"token {os.environ['GITHUB_TOKEN']}"
    return requests.get(url, headers=headers, params=params, timeout=10)

# Smoke test
test = CodeAgent(tools=[], model=model)
print(test.run("Say exactly: ready to match!"))

---
# Section 1 — The A2A Protocol (2 minutes)

Before we build, let's understand the glue that holds everything together.

**A2A (Agent-to-Agent)** — Google's open protocol, April 2025.
Each agent publishes an **Agent Card** and accepts **Tasks**. Any orchestrator can call any agent without knowing its internals.

Think of it as **microservices for AI agents**.

In [ ]:
# ─── A2A Protocol base classes ────────────────────────────────────────────────

class TaskState(Enum):
    SUBMITTED = "submitted"
    WORKING   = "working"
    COMPLETED = "completed"
    FAILED    = "failed"

@dataclass
class A2ATask:
    id: str = field(default_factory=lambda: str(uuid.uuid4())[:8])
    message: str = ""
    context: dict = field(default_factory=dict)
    state: TaskState = TaskState.SUBMITTED
    artifacts: List[dict] = field(default_factory=list)
    error: Optional[str] = None

class A2AAgent:
    @property
    def agent_card(self) -> dict:      raise NotImplementedError
    def _handle(self, task) -> Any:    raise NotImplementedError

    def send_task(self, message: str, context: dict = None) -> A2ATask:
        task = A2ATask(message=message, context=context or {})
        name = self.agent_card['name']
        print(f"  [A2A ->] {name} | task:{task.id}")
        task.state = TaskState.WORKING
        try:
            task.artifacts = [{"type": "data", "data": self._handle(task)}]
            task.state = TaskState.COMPLETED
            print(f"  [A2A <-] {name} | task:{task.id} done")
        except Exception as e:
            task.error, task.state = str(e), TaskState.FAILED
            print(f"  [A2A x]  {name} | failed: {e}")
        return task

print("A2A protocol base ready.")

# Show what an Agent Card looks like
example_card = {
    "name": "RepoFinderAgent",
    "description": "Discovers GitHub repos matching a developer skill profile",
    "version": "1.0.0",
    "capabilities": {"streaming": False, "push_notifications": False},
    "skills": [{"id": "find_repos", "name": "Find Matching Repositories",
                "input_modes": ["text"], "output_modes": ["data"]}]
}
print("\nExample Agent Card:")
print(json.dumps(example_card, indent=2))

---
# Section 2 — The 3 Specialist Agents

Each is an independent A2A agent. In production: 3 separate services. Here: same Python process.

In [ ]:
# ─── Agent 1: RepoFinderAgent ─────────────────────────────────────────────────
class RepoFinderAgent(A2AAgent):
    @property
    def agent_card(self):
        return {"name": "RepoFinderAgent",
                "description": "Finds GitHub repos matching a developer skill profile",
                "version": "1.0.0",
                "skills": [{"id": "find_repos", "input_modes": ["text"], "output_modes": ["data"]}]}

    def _handle(self, task: A2ATask) -> list:
        languages  = task.context.get("languages", ["Python"])
        experience = task.context.get("experience", "intermediate")
        star_range = "100..2000" if experience == "beginner" else "500..50000"
        results = []
        for lang in languages[:2]:
            resp = gh_get("https://api.github.com/search/repositories",
                          params={"q": f"language:{lang} good-first-issues:>3 stars:{star_range}",
                                  "sort": "updated", "per_page": 5})
            if resp.status_code == 200:
                for r in resp.json().get("items", []):
                    results.append({"full_name":   r["full_name"],
                                    "description": (r.get("description") or "")[:100],
                                    "stars":       r["stargazers_count"],
                                    "language":    r.get("language", ""),
                                    "open_issues": r["open_issues_count"]})
        return results[:6]

# ─── Agent 2: ComplexityRaterAgent ───────────────────────────────────────────
class ComplexityRaterAgent(A2AAgent):
    @property
    def agent_card(self):
        return {"name": "ComplexityRaterAgent",
                "description": "Rates a repo's complexity for new contributors",
                "version": "1.0.0",
                "skills": [{"id": "rate_complexity", "input_modes": ["text"], "output_modes": ["data"]}]}

    def _handle(self, task: A2ATask) -> dict:
        owner, repo = task.context["owner"], task.context["repo"]
        d  = gh_get(f"https://api.github.com/repos/{owner}/{repo}").json()
        ld = gh_get(f"https://api.github.com/repos/{owner}/{repo}/languages").json()
        stars, forks = d.get("stargazers_count", 0), d.get("forks_count", 0)
        n = len(ld) if isinstance(ld, dict) else 1
        score = (2 if stars > 10000 else 1 if stars > 2000 else 0) + \
                (2 if n > 4 else 1 if n > 2 else 0) + (1 if forks > 1000 else 0)
        if score <= 1:   c, tip = "beginner-friendly", "Small codebase. Great starting point."
        elif score <= 3: c, tip = "intermediate", "Moderate complexity. Some experience helpful."
        else:            c, tip = "advanced", "Large project. Better after a few contributions elsewhere."
        return {"repo": f"{owner}/{repo}", "complexity": c, "tip": tip, "stars": stars,
                "languages": list(ld.keys())[:5] if isinstance(ld, dict) else []}

# ─── Inner tools for OnboardingGuideAgent ─────────────────────────────────────
@tool
def fetch_contributing_guide(owner: str, repo: str) -> str:
    """Fetches the CONTRIBUTING.md from a GitHub repository.

    Args:
        owner: GitHub username or organisation.
        repo: Repository name.
    """
    for path in ["CONTRIBUTING.md", ".github/CONTRIBUTING.md"]:
        r = gh_get(f"https://api.github.com/repos/{owner}/{repo}/contents/{path}")
        if r.status_code == 200:
            raw = base64.b64decode(r.json()["content"]).decode("utf-8", errors="ignore")
            return re.sub(r'[`#*<>!]', '', raw).strip()[:1000]
    return "No CONTRIBUTING.md found. Check the README for contribution guidelines."

@tool
def fetch_starter_issues(owner: str, repo: str) -> str:
    """Fetches open 'good first issue' and 'help wanted' issues from a GitHub repository.

    Args:
        owner: GitHub username or organisation.
        repo: Repository name.
    """
    found = []
    for label in ["good first issue", "help wanted"]:
        r = gh_get(f"https://api.github.com/repos/{owner}/{repo}/issues",
                   params={"labels": label, "state": "open", "per_page": 3})
        if r.status_code == 200:
            for i in r.json():
                if isinstance(i, dict):
                    found.append(f"[{label}] #{i['number']}: {i['title']}")
    return "\n".join(found) if found else "No labelled starter issues — browse open issues on GitHub."

# ─── Agent 3: OnboardingGuideAgent (wraps smolagents CodeAgent!) ─────────────
class OnboardingGuideAgent(A2AAgent):
    """
    An A2A agent that internally runs a smolagents CodeAgent.
    Nested composition: LangGraph -> A2A -> smolagents -> tools -> GitHub API
    """
    def __init__(self, llm_model):
        self._inner = CodeAgent(tools=[fetch_contributing_guide, fetch_starter_issues],
                                model=llm_model)

    @property
    def agent_card(self):
        return {"name": "OnboardingGuideAgent",
                "description": "Personalised first-contribution guide using an inner smolagents CodeAgent",
                "version": "1.0.0",
                "skills": [{"id": "generate_onboarding", "input_modes": ["text"], "output_modes": ["text"]}]}

    def _handle(self, task: A2ATask) -> str:
        owner  = task.context["owner"]
        repo   = task.context["repo"]
        skills = task.context.get("skills", ["Python"])
        return self._inner.run(
            f"Generate a beginner-friendly first-contribution guide for {owner}/{repo}. "
            f"Developer skills: {', '.join(skills)}. "
            "Use tools to fetch the contributing guide and starter issues. "
            "Return: 1) Setup steps (3 bullets), 2) Best first issue to tackle + why, "
            "3) Which files will likely need editing."
        )

print("All 3 A2A agents defined.")

---
# Section 3 — LangGraph Orchestration

Now wire the three agents into a state graph. Every routing decision is deterministic Python — **no LLM involved in the orchestration layer**.

In [ ]:
class MatchmakerState(TypedDict):
    github_url:        str
    interests:         str
    username:          str
    skill_profile:     dict
    candidate_repos:   list
    rated_repos:       list
    onboarding_guides: list
    contribution_plan: str

def analyze_profile_node(s):
    # Parsing the username out of the URL is trivial, so it lives right here
    # instead of its own node -- no need for a dedicated parsing step.
    username = s["github_url"].rstrip("/").replace("https://github.com/", "").split("/")[0]

    r = gh_get(f"https://api.github.com/users/{username}/repos",
               params={"sort": "updated", "per_page": 15})
    repos = r.json() if r.status_code == 200 else []
    lc = {}
    for repo in repos:
        lang = repo.get("language")
        if lang: lc[lang] = lc.get(lang, 0) + 1
    top = sorted(lc, key=lc.get, reverse=True)[:3] or ["Python"]
    n   = len(repos)
    exp = "beginner" if n < 6 else "intermediate" if n < 20 else "experienced"
    profile = {"username": username, "top_languages": top, "experience": exp,
               "interests": [i.strip() for i in s.get("interests","").split(",") if i.strip()]}
    print(f"[LangGraph] username = {username}")
    print(f"[LangGraph] Profile: {top} | {exp}")
    return {"username": username, "skill_profile": profile}

def repo_discovery_node(s):
    print("[LangGraph] -> RepoFinderAgent")
    task = RepoFinderAgent().send_task("Find repos",
               context={"languages": s["skill_profile"]["top_languages"],
                        "experience": s["skill_profile"]["experience"]})
    return {"candidate_repos": task.artifacts[0]["data"] if task.state == TaskState.COMPLETED else []}

def complexity_rating_node(s):
    print("[LangGraph] -> ComplexityRaterAgent")
    rater, rated = ComplexityRaterAgent(), []
    for repo in s["candidate_repos"][:4]:
        owner, rname = repo["full_name"].split("/")
        task = rater.send_task(f"Rate {repo['full_name']}", context={"owner": owner, "repo": rname})
        if task.state == TaskState.COMPLETED:
            r = task.artifacts[0]["data"]
            r["description"] = repo.get("description", "")
            rated.append(r)
    target  = "beginner-friendly" if s["skill_profile"]["experience"] == "beginner" else "intermediate"
    matched = [r for r in rated if r["complexity"] == target] or rated
    return {"rated_repos": matched}

def onboarding_node(s):
    print("[LangGraph] -> OnboardingGuideAgent (smolagents CodeAgent inside)")
    guide_agent = OnboardingGuideAgent(model)
    skills, guides = s["skill_profile"]["top_languages"], []
    for repo in s["rated_repos"][:2]:
        owner, rname = repo["repo"].split("/")
        task = guide_agent.send_task(f"Onboard into {repo['repo']}",
                                     context={"owner": owner, "repo": rname, "skills": skills})
        if task.state == TaskState.COMPLETED:
            guides.append({"repo": repo["repo"], "complexity": repo["complexity"],
                           "tip": repo["tip"], "guide": task.artifacts[0]["data"]})
    return {"onboarding_guides": guides}

def format_plan_node(s):
    p, guides = s["skill_profile"], s["onboarding_guides"]
    lines = [
        "=" * 56,
        f"  YOUR FIRST PR PLAN  --  @{p['username']}",
        "=" * 56,
        f"  Skills detected : {', '.join(p['top_languages'])}",
        f"  Level           : {p['experience']}",
        ""
    ]
    for i, g in enumerate(guides, 1):
        lines += [f"{'─'*56}",
                  f"  Opportunity {i}: {g['repo']}  [{g['complexity']}]",
                  f"  {g['tip']}", f"{'─'*56}",
                  g["guide"], ""]
    lines += ["─"*56,
              "Powered by: LangGraph + A2A + smolagents (Llama 3.3 on Groq)"]
    return {"contribution_plan": "\n".join(lines)}

# Build the graph
b = StateGraph(MatchmakerState)
for name, fn in [("analyze_profile",analyze_profile_node),
                 ("repo_discovery",repo_discovery_node),("complexity_rating",complexity_rating_node),
                 ("onboarding",onboarding_node),("format_plan",format_plan_node)]:
    b.add_node(name, fn)
b.set_entry_point("analyze_profile")
for a, z in [("analyze_profile","repo_discovery"),
             ("repo_discovery","complexity_rating"),("complexity_rating","onboarding"),
             ("onboarding","format_plan"),("format_plan",END)]:
    b.add_edge(a, z)

matchmaker = b.compile()

# Show the graph
print(matchmaker.get_graph().draw_mermaid())

---
# Section 4 — Live Demo Run

First, the presenter runs it with their own profile.

In [ ]:
def run_matchmaker(github_url: str, interests: str = "") -> str:
    result = matchmaker.invoke({
        "github_url":        github_url,
        "interests":         interests,
        "username":          "",
        "skill_profile":     {},
        "candidate_repos":   [],
        "rated_repos":       [],
        "onboarding_guides": [],
        "contribution_plan": ""
    })
    return result["contribution_plan"]

# Presenter runs this first
plan = run_matchmaker(
    github_url = "https://github.com/ajmalbinnizam",
    interests  = "data engineering, Python, APIs"
)
print(plan)

---
# Section 5 — Your Turn!

> **Audience:** Just enter your GitHub **username** below (not the full URL) and run the cell.
> You'll get a personalised "Your First PR" plan in about 30 seconds.
>
> **Don't have a GitHub account?** Leave the username blank — the cell will pick a well-known
> open-source profile for you to try the demo with instead.

```
Ideas for interests:
  "web, APIs, REST"
  "data, pandas, ML"
  "DevOps, Docker, CI"
  "frontend, React, TypeScript"
  "security, cryptography"
```

In [ ]:
# ─── Just fill this in ─────────────────────────────────────────────────────
GITHUB_USERNAME = ""          # e.g. "torvalds"  -- just the username, not the full URL
MY_INTERESTS    = "web, APIs" # comma-separated topics
# ─────────────────────────────────────────────────────────────────────────────

# No GitHub account? Leave GITHUB_USERNAME blank -- we'll pick a sample profile for you.
SAMPLE_USERNAMES = ["torvalds", "gvanrossum", "yyx990803", "sindresorhus", "gaearon"]

if not GITHUB_USERNAME.strip():
    import random
    GITHUB_USERNAME = random.choice(SAMPLE_USERNAMES)
    print(f"No username entered — using a sample profile instead: {GITHUB_USERNAME}\n")

github_url = f"https://github.com/{GITHUB_USERNAME}"
print(run_matchmaker(github_url, MY_INTERESTS))

---
## What Just Happened — Step by Step

```
Your GitHub URL
  │
  ▼  [LangGraph] analyze_profile_node
     Extracted username (one line — no separate parsing node needed),
     called GitHub API, counted your repo languages,
     estimated your experience level
  │
  ▼  [A2A] RepoFinderAgent.send_task()
     Searched GitHub for repos matching your languages + level
     Returns: list of candidate repos
  │
  ▼  [A2A] ComplexityRaterAgent.send_task()  (once per repo)
     Python heuristic: stars + languages + forks = complexity score
     Filtered to repos that match YOUR experience
  │
  ▼  [A2A] OnboardingGuideAgent.send_task()
     Internally ran: smolagents CodeAgent.run()
     The LLM wrote Python to call:
       fetch_contributing_guide(owner, repo)  -> CONTRIBUTING.md
       fetch_starter_issues(owner, repo)       -> open issues
     AST sandbox executed the code safely
     LLM synthesised: setup steps + best issue + files to edit
  │
  ▼  [LangGraph] format_plan_node
     String formatting — no LLM call
```

| Step | Framework | What happened | LLM call? |
|------|-----------|---------------|-----------|
| analyze_profile | LangGraph | Extracted username from URL, fetched repos, built skill profile | No |
| repo_discovery | A2A RepoFinderAgent | GitHub API search | No |
| complexity_rating | A2A ComplexityRaterAgent | Python heuristic on repo metadata | No |
| onboarding | A2A OnboardingGuideAgent + smolagents CodeAgent | LLM wrote Python, called 2 tools | **Yes — once per repo** |
| format_plan | LangGraph | String formatting | No |

**LLM calls: 1 per repo** (inside OnboardingGuideAgent).  
**Everything else: deterministic Python** — routing, filtering, formatting, API calls.

---
## Resources

| Resource | Link |
|----------|------|
| smolagents docs | https://huggingface.co/docs/smolagents |
| LangGraph docs | https://langchain-ai.github.io/langgraph |
| A2A Protocol | https://github.com/google/A2A |
| Groq free API | https://console.groq.com |
| Study notebooks | `01_smolagents_basics` → `02_langgraph_basics` → `03_a2a_basics` |

**Questions? Find us after the talk!**